# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 03_fastai/05_titanic_models_from_scratch.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '05_titanic_models_from_scratch.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 5: linear and deep models from scratch
Source task: https://github.com/fastai/course22/blob/master/05-linear-model-and-neural-net-from-scratch.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record. Imputation, logarithmic fare transformation, one-hot encoding and scaling are fitted on training rows. The public seaborn Titanic teaching copy supplies the core features; local original Kaggle CSVs take priority.


In [ ]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

from torch import nn
from sklearn.metrics import accuracy_score,f1_score
from fastai_utils import *
frame,tr,va,processor,features,target=split_and_prepare()
xx=torch.tensor(features);yy=torch.tensor(target).reshape(-1,1)
print('Shapes:',xx.shape,yy.shape)
def initialise(sizes):
    return [(torch.randn(a,b)*np.sqrt(2/a)).requires_grad_() for a,b in zip(sizes[:-1],sizes[1:])], [torch.zeros(1,b,requires_grad=True) for b in sizes[1:]]
def logits(weights,biases,inputs):
    h=inputs
    for index,(weight,bias) in enumerate(zip(weights,biases)):
        h=h@weight+bias
        if index<len(weights)-1:h=torch.relu(h)
    return h
def train_scratch(sizes,epochs=250,lr=.05):
    torch.manual_seed(42);weights,biases=initialise(sizes);history=[]
    for epoch in range(epochs):
        prediction=logits(weights,biases,xx[tr])
        loss=nn.functional.binary_cross_entropy_with_logits(prediction,yy[tr]);loss.backward()
        with torch.no_grad():
            for p in weights+biases:
                p-=lr*p.grad;p.grad.zero_()
        history.append(float(loss.detach()))
    return weights,biases,history
models={};rows=[]
for name,hidden in [('linear',[]),('one_hidden_layer',[20]),('deep',[20,10])]:
    w,b,history=train_scratch([xx.shape[1]]+hidden+[1])
    with torch.no_grad():p=logits(w,b,xx[va]).sigmoid().ravel().numpy()
    rows.append({'model':name,'validation_accuracy':accuracy_score(target[va],p>=.5),
        'validation_macro_f1':f1_score(target[va],p>=.5,average='macro')})
    models[name]=(w,b)
    pd.DataFrame({'epoch':range(len(history)),'bce_loss':history}).to_csv(results/f'titanic_{name}_history.csv',index=False)
table=pd.DataFrame(rows);print(table.to_string(index=False));table.to_csv(results/'titanic_scratch_comparison.csv',index=False)

In [ ]:
selected=table.loc[table.validation_macro_f1.idxmax(),'model'];w,b=models[selected]
test_path=Path('data/titanic/test.csv')
if test_path.exists():
    test_frame=add_features(pd.read_csv(test_path));test_x=torch.tensor(processor.transform(test_frame[CONT+CAT]),dtype=torch.float32)
    with torch.no_grad():pred=(logits(w,b,test_x).sigmoid().ravel()>=.5).int().numpy()
    submission=pd.DataFrame({'PassengerId':test_frame.PassengerId,'Survived':pred})
    submission.to_csv(results/'titanic_submission.csv',index=False)
    print('Created local submission CSV; nothing was submitted to Kaggle.')
else:
    with torch.no_grad():pred=logits(w,b,xx[va]).sigmoid().ravel().numpy()
    pd.DataFrame({'source_row':va,'true_survived':target[va],'probability':pred}).to_csv(results/'titanic_validation_predictions.csv',index=False)
    print('Kaggle test.csv was not supplied. Saved validation predictions; competition export code is complete but unrun.')

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)